In [3]:
# ==================================================
# IST3134 Big Data Analytics Assignment
# Apache Spark Implementation
# Video Game Market Price and Revenue Dataset
# ==================================================

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
start_time = time.time()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
dataset_path = "s3://ist3134-videogame-bigdata-2026/input/videogames_processed.csv"

print(dataset_path)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s3://ist3134-videogame-bigdata-2026/input/videogames_processed.csv

In [7]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(dataset_path)
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
print("Number of columns:", len(df.columns))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Number of columns: 130

In [9]:
df.columns

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['game_id', 'platform', 'platform_code', 'storefront', 'obs_date', 'days_since_release', 'current_price_usd', 'original_price_usd', 'discount_pct', 'is_on_sale', 'sale_event_name', 'lowest_price_usd', 'decay_factor', 'price_vs_launch', 'concurrent_players', 'peak_ccu_alltime', 'review_count', 'positive_reviews', 'negative_reviews', 'steam_rating_pct', 'user_score', 'metacritic_score', 'wishlist_count', 'twitch_viewers_proxy', 'youtube_views_proxy', 'reddit_mentions', 'social_hype_index', 'has_controversy', 'controversy_day', 'estimated_units_sold', 'estimated_revenue_usd', 'revenue_per_review', 'revenue_per_player', 'price_na', 'price_eu', 'price_gb', 'price_jp', 'price_br', 'price_au', 'game_name', 'developer', 'publisher', 'publisher_tier', 'franchise', 'genre', 'genre_code', 'subgenre', 'tags', 'age_rating', 'release_date', 'release_year', 'release_month', 'release_quarter', 'base_price_usd', 'dlc_count', 'expansion_count', 'achievement_count', 'supported_languages', 'is_free_to_pla

In [10]:
df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- game_id: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- platform_code: string (nullable = true)
 |-- storefront: string (nullable = true)
 |-- obs_date: date (nullable = true)
 |-- days_since_release: double (nullable = true)
 |-- current_price_usd: double (nullable = true)
 |-- original_price_usd: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- is_on_sale: double (nullable = true)
 |-- sale_event_name: string (nullable = true)
 |-- lowest_price_usd: double (nullable = true)
 |-- decay_factor: double (nullable = true)
 |-- price_vs_launch: double (nullable = true)
 |-- concurrent_players: double (nullable = true)
 |-- peak_ccu_alltime: double (nullable = true)
 |-- review_count: double (nullable = true)
 |-- positive_reviews: double (nullable = true)
 |-- negative_reviews: double (nullable = true)
 |-- steam_rating_pct: double (nullable = true)
 |-- user_score: double (nullable = true)
 |-- metacritic_score: double (nullable = 

In [11]:
df.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+---------------+-------------+--------------+----------+------------------+-----------------+------------------+------------+----------+---------------+----------------+------------+---------------+------------------+----------------+------------+----------------+----------------+----------------+----------+----------------+--------------+--------------------+-------------------+---------------+-----------------+---------------+---------------+--------------------+---------------------+------------------+------------------+--------+--------+--------+--------+--------+--------+---------------+-------------------+-------------------+--------------+---------+------+----------+------------+---------------------------------+----------+------------+------------+-------------+---------------+--------------+---------+---------------+-----------------+-------------------+---------------+--------+---------------+--------------+---------+------------+---------+-----------+---------+---

In [12]:
# Select only columns required for the analysis
selected_df = df.select(
    "game_id",
    "platform",
    "genre",
    "obs_date",
    "current_price_usd",
    "revenue_cumulative"
)

selected_df.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+---------------+------+----------+-----------------+------------------+
|game_id  |platform       |genre |obs_date  |current_price_usd|revenue_cumulative|
+---------+---------------+------+----------+-----------------+------------------+
|G00000000|Nintendo Switch|Horror|2019-01-11|33.99            |1.745376E7        |
|G00000000|Nintendo Switch|Horror|2019-01-17|33.99            |3.0103975E7       |
|G00000000|Nintendo Switch|Horror|2019-01-21|33.99            |4.4087026E7       |
|G00000000|Nintendo Switch|Horror|2019-01-23|33.99            |5.4658066E7       |
|G00000000|Nintendo Switch|Horror|2019-01-30|33.99            |6.6289155E7       |
+---------+---------------+------+----------+-----------------+------------------+
only showing top 5 rows

In [13]:
# Count the complete dataset
import time

count_start = time.time()

raw_row_count = df.count()

count_end = time.time()

print("Raw number of rows:", raw_row_count)
print("Row count execution time:", round(count_end - count_start, 2), "seconds")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Raw number of rows: 5584706
Row count execution time: 18.37 seconds

In [14]:
# Check the number of partitions
print("Number of Spark partitions:", selected_df.rdd.getNumPartitions())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Number of Spark partitions: 38

In [16]:
# Check missing value
selected_df.select(
    [
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in selected_df.columns
    ]
).show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+--------+-----+--------+-----------------+------------------+
|game_id|platform|genre|obs_date|current_price_usd|revenue_cumulative|
+-------+--------+-----+--------+-----------------+------------------+
|      0|       0|    0|       0|                0|                 0|
+-------+--------+-----+--------+-----------------+------------------+

In [17]:
# Remove unusable records
clean_df = selected_df.dropna(
    subset=[
        "game_id",
        "platform",
        "genre",
        "obs_date",
        "current_price_usd",
        "revenue_cumulative"
    ]
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
# Check how many records remain
clean_row_count = clean_df.count()

print("Rows before null removal:", raw_row_count)
print("Rows after null removal:", clean_row_count)
print("Rows removed:", raw_row_count - clean_row_count)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Rows before null removal: 5584706
Rows after null removal: 5584706
Rows removed: 0

In [19]:
# Check exact duplicate rows
duplicate_count = clean_df.count() - clean_df.dropDuplicates().count()

print("Exact duplicate rows:", duplicate_count)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exact duplicate rows: 0

In [20]:
# Remove exact duplicates
clean_df = clean_df.dropDuplicates()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
# Import the Spark window function
from pyspark.sql.window import Window

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [22]:
latest_window = (
    Window
    .partitionBy("game_id", "platform")
    .orderBy(F.col("obs_date").desc())
)

latest_df = (
    clean_df
    .withColumn(
        "row_number",
        F.row_number().over(latest_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
# Count the latest records
latest_row_count = latest_df.count()

print("Clean observation-level rows:", clean_df.count())
print("Latest game-platform records:", latest_row_count)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Clean observation-level rows: 5584706
Latest game-platform records: 28634

In [24]:
# Preview the results
latest_df.orderBy("game_id", "platform").show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+---------------+-------+----------+-----------------+------------------+
|game_id  |platform       |genre  |obs_date  |current_price_usd|revenue_cumulative|
+---------+---------------+-------+----------+-----------------+------------------+
|G00000000|Nintendo Switch|Horror |2024-12-29|7.99             |3.90991955E9      |
|G00000000|Xbox One       |Horror |2024-12-30|8.99             |3.936902101E9     |
|G00000001|Xbox Series X/S|Puzzle |2024-12-27|17.99            |5.255094369E10    |
|G00000002|PlayStation 5  |Racing |2024-12-27|27.99            |1.1386476E7       |
|G00000002|Xbox One       |Racing |2024-12-30|22.99            |1.4697661E7       |
|G00000003|PlayStation 5  |Sandbox|2024-12-28|4.99             |9.4161299E7       |
|G00000004|Nintendo Switch|Puzzle |2024-12-18|8.99             |5.1735086E7       |
|G00000004|PC             |Puzzle |2024-12-12|9.99             |3.1529258E7       |
|G00000004|PlayStation 4  |Puzzle |2024-12-27|1.99             |2.1161137E7 

In [25]:
# Cache the prepared dataset
latest_df = latest_df.cache()

latest_df.count()

print("Prepared dataset cached successfully.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Prepared dataset cached successfully.

In [30]:
print("========================================")
print("Apache Spark Big Data Analytics")
print("Dataset:", dataset_path)
print("Rows:", raw_row_count)
print("Columns:", len(df.columns))
print("Prepared Dataset:", latest_row_count)
print("========================================")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Apache Spark Big Data Analytics
Dataset: s3://ist3134-videogame-bigdata-2026/input/videogames_processed.csv
Rows: 5584706
Columns: 130
Prepared Dataset: 28634

In [26]:
# ============================================
# Analysis 1 - Total Revenue by Genre
# ============================================

genre_start = time.time()

revenue_by_genre = (
    latest_df
    .groupBy("genre")
    .agg(
        F.round(
            F.sum("revenue_cumulative"), 2
        ).alias("total_revenue_usd"),

        F.count("*").alias("number_of_games")
    )
    .orderBy(F.desc("total_revenue_usd"))
)

revenue_by_genre.show(50, truncate=False)

genre_end = time.time()

print(
    "Execution Time:",
    round(genre_end - genre_start,2),
    "seconds"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+------------------+---------------+
|genre               |total_revenue_usd |number_of_games|
+--------------------+------------------+---------------+
|Action              |4.9852971901246E13|3596           |
|RPG                 |4.5087152335603E13|2905           |
|First-Person Shooter|3.7285531893895E13|2368           |
|Adventure           |2.9968344275258E13|2156           |
|Sports              |2.564146266568E13 |1808           |
|Strategy            |2.1448687157402E13|2032           |
|Fighting            |1.7848275441578E13|1304           |
|Platformer          |1.6988041478653E13|1596           |
|Simulation          |1.6045812959844E13|1825           |
|Racing              |1.5360686041622E13|1074           |
|Horror              |1.0503302009503E13|1315           |
|Sandbox             |8.445222246828E12 |1073           |
|Puzzle              |7.108144416037E12 |1206           |
|Indie               |6.306370896171E12 |1642           |
|Visual Novel 

In [27]:
# ============================================
# Analysis 2 - Total Revenue by Platform
# ============================================

platform_start = time.time()

revenue_by_platform = (
    latest_df
    .groupBy("platform")
    .agg(
        F.round(
            F.sum("revenue_cumulative"),2
        ).alias("total_revenue_usd"),

        F.count("*").alias("number_of_games")
    )
    .orderBy(F.desc("total_revenue_usd"))
)

revenue_by_platform.show(50, truncate=False)

platform_end = time.time()

print(
    "Execution Time:",
    round(platform_end-platform_start,2),
    "seconds"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+------------------+---------------+
|platform       |total_revenue_usd |number_of_games|
+---------------+------------------+---------------+
|PlayStation 5  |5.6420574157653E13|4735           |
|PC             |5.5067436940648E13|4802           |
|Xbox Series X/S|5.4238133475203E13|4760           |
|Nintendo Switch|5.3970525123124E13|4806           |
|Xbox One       |4.7975186913562E13|4850           |
|PlayStation 4  |4.7583541439572E13|4681           |
+---------------+------------------+---------------+

Execution Time: 2.71 seconds

In [28]:
# ============================================
# Analysis 3 - Average Price by Genre
# ============================================

price_start = time.time()

average_price_by_genre = (
    latest_df
    .groupBy("genre")
    .agg(
        F.round(
            F.avg("current_price_usd"),2
        ).alias("average_price_usd"),

        F.count("*").alias("number_of_games")
    )
    .orderBy(F.desc("average_price_usd"))
)

average_price_by_genre.show(50, truncate=False)

price_end = time.time()

print(
    "Execution Time:",
    round(price_end-price_start,2),
    "seconds"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-----------------+---------------+
|genre               |average_price_usd|number_of_games|
+--------------------+-----------------+---------------+
|Sports              |37.78            |1808           |
|First-Person Shooter|37.34            |2368           |
|RPG                 |37.32            |2905           |
|Action              |31.93            |3596           |
|Fighting            |31.71            |1304           |
|Adventure           |31.4             |2156           |
|Racing              |30.81            |1074           |
|Platformer          |23.63            |1596           |
|Strategy            |23.08            |2032           |
|Simulation          |21.93            |1825           |
|Sandbox             |19.15            |1073           |
|Horror              |19.07            |1315           |
|Puzzle              |11.63            |1206           |
|Indie               |9.13             |1642           |
|Visual Novel        |9.13     

In [29]:
# ============================================
# Analysis 4 - Summary Table
# ============================================

summary_start = time.time()

summary_table = (
    latest_df
    .groupBy("genre","platform")
    .agg(
        F.round(
            F.sum("revenue_cumulative"),2
        ).alias("total_revenue_usd"),

        F.round(
            F.avg("current_price_usd"),2
        ).alias("average_price_usd"),

        F.count("*").alias("number_of_games")
    )
    .orderBy(F.desc("total_revenue_usd"))
)

summary_table.show(50,truncate=False)

summary_end = time.time()

print(
    "Execution Time:",
    round(summary_end-summary_start,2),
    "seconds"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+---------------+-----------------+-----------------+---------------+
|genre               |platform       |total_revenue_usd|average_price_usd|number_of_games|
+--------------------+---------------+-----------------+-----------------+---------------+
|RPG                 |PlayStation 5  |8.911306987686E12|37.21            |503            |
|Action              |PC             |8.898724989247E12|31.11            |576            |
|Action              |PlayStation 5  |8.750693293731E12|35.86            |593            |
|Action              |Xbox Series X/S|8.74939304921E12 |34.82            |600            |
|Action              |Nintendo Switch|8.510849764513E12|32.46            |618            |
|RPG                 |PC             |7.899182760885E12|38.71            |492            |
|RPG                 |Nintendo Switch|7.6559476954E12  |38.91            |485            |
|Action              |PlayStation 4  |7.578400664765E12|29.24            |612            |

In [2]:
end_time = time.time()

print(f"Total Execution Time: {end_time - start_time:.2f} seconds")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total Execution Time: 35.03 seconds